In [1]:
import pandas as pd
import numpy as np
import pathlib
import os
import pickle
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
import pathlib

from final_project_package.ml_logic.data_clean import initialize_clip, data_clean, add_clip_columns, average_scoring, parse_layout
from final_project_package.ml_logic.model import initialize_model, train_model, evaluate_model
from final_project_package.ml_logic.preprocessor_pipeline import get_fitted_preprocessor


✅ TensorFlow loaded (0.03s)


In [2]:
#load dataset(double check name of the main dataframe)
data_path = pathlib.Path(".")
data = pd.read_csv(data_path / "data_dump/listings_with_scores.csv")
# image_data = pd.read_csv(data_path / 'raw_data/images.csv')
data


,index,source_id,url,price_man_yen,area_sqm,year_built,floor_number,floors_total,address,nearest_station,...,brightness_kitchen,brightness_living_room,brightness_toilet,condition_bathroom,condition_bedroom,condition_kitchen,condition_living_room,condition_toilet,rooms_num,base_layout
0,0,20277732,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,2390,52.89,1989,4,4,東京都足立区六月１ [ ■ 周辺環境 ],竹ノ塚駅,...,0.226562,NaN,0.132812,0.401367,0.203125,0.531250,NaN,0.246094,2,LDK
1,1,20332918,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,1580,29.16,1976,6,9,東京都足立区東和５ [ ■ 周辺環境 ],北綾瀬駅,...,0.495117,NaN,0.183594,0.378906,0.539062,0.621094,NaN,0.320312,1,DK
2,2,20344616,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,2380,53.04,1974,3,8,東京都足立区梅田７-22-9 [ ■ 周辺環境 ],梅島駅,...,0.347656,NaN,0.042969,0.255208,0.462891,0.468750,NaN,0.203125,2,LDK
3,3,78385243,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_7...,1700,40.10,1985,1,5,東京都足立区青井１-16-9 [ ■ 周辺環境 ],青井駅,...,NaN,NaN,NaN,0.394531,0.634766,NaN,NaN,NaN,2,DK
4,4,78151350,https://suumo.jp/ms/chuko/tokyo/sc_bunkyo/nc_7...,1800,26.29,1977,5,5,東京都文京区小石川５ [ ■ 周辺環境 ],茗荷谷駅,...,0.367188,NaN,NaN,0.281250,0.621094,0.658203,NaN,NaN,1,DK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3348,3348,78637031,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,118000,106.76,2024,5,2,東京都渋谷区神宮前６ [ ■ 周辺環境 ],明治神宮前駅,...,0.281250,0.287109,NaN,0.417969,0.677083,0.626953,0.733398,NaN,2,LDK
3349,3349,78783232,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,79800,152.01,2014,2,1,東京都渋谷区広尾３-12-17 [ ■ 周辺環境 ],広尾駅,...,0.377604,0.366406,NaN,NaN,NaN,0.764323,0.632031,NaN,3,LDK
3350,3350,78931650,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,137000,214.98,2024,4,1,東京都渋谷区鉢山町 [ ■ 周辺環境 ],代官山駅,...,0.562500,0.392578,0.468750,0.369141,0.816406,0.851562,0.824219,0.378906,3,LDK
3351,3351,79115894,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,79800,177.80,2025,2,4,東京都渋谷区西原３ [ ■ 周辺環境 ],代々木上原駅,...,0.531250,0.378906,NaN,0.586719,0.761719,0.638672,0.621094,NaN,2,LDK


In [3]:
X = data.drop(columns=["price_man_yen"]).copy()
y = data["price_man_yen"]

In [4]:
X

,index,source_id,url,area_sqm,year_built,floor_number,floors_total,address,nearest_station,walk_minutes,...,brightness_kitchen,brightness_living_room,brightness_toilet,condition_bathroom,condition_bedroom,condition_kitchen,condition_living_room,condition_toilet,rooms_num,base_layout
0,0,20277732,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,52.89,1989,4,4,東京都足立区六月１ [ ■ 周辺環境 ],竹ノ塚駅,16,...,0.226562,NaN,0.132812,0.401367,0.203125,0.531250,NaN,0.246094,2,LDK
1,1,20332918,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,29.16,1976,6,9,東京都足立区東和５ [ ■ 周辺環境 ],北綾瀬駅,12,...,0.495117,NaN,0.183594,0.378906,0.539062,0.621094,NaN,0.320312,1,DK
2,2,20344616,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_2...,53.04,1974,3,8,東京都足立区梅田７-22-9 [ ■ 周辺環境 ],梅島駅,7,...,0.347656,NaN,0.042969,0.255208,0.462891,0.468750,NaN,0.203125,2,LDK
3,3,78385243,https://suumo.jp/ms/chuko/tokyo/sc_adachi/nc_7...,40.10,1985,1,5,東京都足立区青井１-16-9 [ ■ 周辺環境 ],青井駅,10,...,NaN,NaN,NaN,0.394531,0.634766,NaN,NaN,NaN,2,DK
4,4,78151350,https://suumo.jp/ms/chuko/tokyo/sc_bunkyo/nc_7...,26.29,1977,5,5,東京都文京区小石川５ [ ■ 周辺環境 ],茗荷谷駅,9,...,0.367188,NaN,NaN,0.281250,0.621094,0.658203,NaN,NaN,1,DK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3348,3348,78637031,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,106.76,2024,5,2,東京都渋谷区神宮前６ [ ■ 周辺環境 ],明治神宮前駅,3,...,0.281250,0.287109,NaN,0.417969,0.677083,0.626953,0.733398,NaN,2,LDK
3349,3349,78783232,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,152.01,2014,2,1,東京都渋谷区広尾３-12-17 [ ■ 周辺環境 ],広尾駅,12,...,0.377604,0.366406,NaN,NaN,NaN,0.764323,0.632031,NaN,3,LDK
3350,3350,78931650,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,214.98,2024,4,1,東京都渋谷区鉢山町 [ ■ 周辺環境 ],代官山駅,10,...,0.562500,0.392578,0.468750,0.369141,0.816406,0.851562,0.824219,0.378906,3,LDK
3351,3351,79115894,https://suumo.jp/ms/chuko/tokyo/sc_shibuya/nc_...,177.80,2025,2,4,東京都渋谷区西原３ [ ■ 周辺環境 ],代々木上原駅,5,...,0.531250,0.378906,NaN,0.586719,0.761719,0.638672,0.621094,NaN,2,LDK


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)
preprocesser = get_fitted_preprocessor(X_train)


Preprocessing features...
✅ returned preprocessor


In [6]:
preprocesser.get_feature_names_out()

array(['keep_columns__rooms_num', 'num_transformer__area_sqm',
       'num_transformer__year_built', 'num_transformer__floor_number',
       'num_transformer__floors_total', 'num_transformer__walk_minutes',
       'nearest_station_tranformer__nearest_station_三軒茶屋駅',
       'nearest_station_tranformer__nearest_station_上野駅',
       'nearest_station_tranformer__nearest_station_中目黒駅',
       'nearest_station_tranformer__nearest_station_二子玉川駅',
       'nearest_station_tranformer__nearest_station_五反田駅',
       'nearest_station_tranformer__nearest_station_代官山駅',
       'nearest_station_tranformer__nearest_station_住吉駅',
       'nearest_station_tranformer__nearest_station_入谷駅',
       'nearest_station_tranformer__nearest_station_八丁堀駅',
       'nearest_station_tranformer__nearest_station_勝どき駅',
       'nearest_station_tranformer__nearest_station_北千住駅',
       'nearest_station_tranformer__nearest_station_北綾瀬駅',
       'nearest_station_tranformer__nearest_station_千歳船橋駅',
       'nearest_station_tr

In [7]:
X_train_preprocessed = pd.DataFrame(preprocesser.transform(X_train), columns=preprocesser.get_feature_names_out(), index=X_train.index)

In [20]:
X_train_preprocessed

,keep_columns__rooms_num,num_transformer__area_sqm,num_transformer__year_built,num_transformer__floor_number,num_transformer__floors_total,num_transformer__walk_minutes,nearest_station_tranformer__nearest_station_三軒茶屋駅,nearest_station_tranformer__nearest_station_上野駅,nearest_station_tranformer__nearest_station_中目黒駅,nearest_station_tranformer__nearest_station_二子玉川駅,...,nearest_station_tranformer__nearest_station_錦糸町駅,nearest_station_tranformer__nearest_station_飯田橋駅,nearest_station_tranformer__nearest_station_駒沢大学駅,nearest_station_tranformer__nearest_station_高田馬場駅,nearest_station_tranformer__nearest_station_麻布十番駅,nearest_station_tranformer__nearest_station_infrequent_sklearn,ordinal_transformer__base_layout,mean_luxury_transformer__mean_luxury,mean_brightness_transformer__mean_brightness,mean_condition_transformer__mean_condition
2461,1.0,-0.782412,0.370370,0.000000,-0.444444,0.166667,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.457031,0.407227,0.568359
875,1.0,-0.726655,0.407407,1.000000,0.777778,-0.666667,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.576128,0.347830,0.593533
2045,3.0,1.730734,-0.518519,-0.428571,-0.555556,1.166667,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.439648,0.411654,0.567187
1197,3.0,-0.382140,-0.814815,-0.142857,0.444444,-0.666667,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.388170,0.408817,0.567076
2496,3.0,-0.016772,0.555556,0.714286,0.888889,-0.333333,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.379123,0.318142,0.499566
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3149,2.0,0.948323,0.037037,2.714286,-0.444444,-0.500000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.546224,0.315039,0.576953
1321,1.0,-1.160925,0.370370,-0.428571,1.000000,-0.833333,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.374039,0.265780,0.536334
1452,1.0,-0.446510,0.370370,-0.142857,-0.555556,-0.500000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,3.0,0.316650,0.159505,0.501546
1178,4.0,1.543064,-0.296296,0.428571,-0.555556,1.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.320964,0.233659,0.586068


In [9]:
y_train

2461    14980
875      5800
2045    11980
1197     6980
2496    14980
        ...  
3149    33980
1321     7480
1452     7980
1178     6980
1981    11490
Name: price_man_yen, Length: 2347, dtype: int64

In [10]:
X_train

,index,source_id,url,area_sqm,year_built,floor_number,floors_total,address,nearest_station,walk_minutes,...,brightness_kitchen,brightness_living_room,brightness_toilet,condition_bathroom,condition_bedroom,condition_kitchen,condition_living_room,condition_toilet,rooms_num,base_layout
2461,2461,79155014,https://suumo.jp/ms/chuko/tokyo/sc_minato/nc_7...,47.64,2012,5,2,東京都港区南青山４ [ ■ 周辺環境 ],乃木坂駅,8,...,0.324219,0.269531,0.796875,0.386719,NaN,0.593750,0.730469,0.562500,1,LDK
875,875,20353844,https://suumo.jp/ms/chuko/tokyo/sc_itabashi/nc...,48.87,2013,12,13,東京都板橋区板橋１-41-8 [ ■ 周辺環境 ],新板橋駅,3,...,0.335938,NaN,NaN,0.489583,0.740234,0.550781,NaN,NaN,1,LDK
2045,2045,78565723,https://suumo.jp/ms/chuko/tokyo/sc_setagaya/nc...,103.08,1988,2,1,東京都世田谷区桜上水２ [ ■ 周辺環境 ],経堂駅,14,...,0.453125,0.591797,0.203125,0.500000,0.585938,0.515625,0.828125,0.406250,3,LDK
1197,1197,20354140,https://suumo.jp/ms/chuko/tokyo/sc_chuo/nc_203...,56.47,1980,4,10,東京都中央区日本橋箱崎町37-4 [ ■ 周辺環境 ],水天宮前駅,3,...,0.529018,0.183594,0.578125,0.621094,0.562500,0.640067,0.468750,0.542969,3,LDK
2496,2496,20055858,https://suumo.jp/ms/chuko/tokyo/sc_taito/nc_20...,64.53,2017,10,14,東京都台東区上野７ [ ■ 周辺環境 ],上野駅,5,...,0.471354,NaN,NaN,0.347656,0.582031,0.569010,NaN,NaN,3,LDK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3149,3149,79212713,https://suumo.jp/ms/chuko/tokyo/sc_chiyoda/nc_...,85.82,2003,24,2,東京都千代田区飯田橋２ [ ■ 周辺環境 ],九段下駅,4,...,0.248698,0.227539,0.796875,0.479167,0.513672,0.598958,0.730469,0.562500,2,LDK
1321,1321,79082645,https://suumo.jp/ms/chuko/tokyo/sc_chiyoda/nc_...,39.29,2012,2,15,東京都千代田区東神田１ [ ■ 周辺環境 ],馬喰町駅,2,...,0.220052,NaN,NaN,0.451823,0.615513,0.541667,NaN,NaN,1,LDK
1452,1452,79138041,https://suumo.jp/ms/chuko/tokyo/sc_koto/nc_791...,55.05,2012,4,1,東京都江東区東雲２ [ ■ 周辺環境 ],東雲駅,4,...,0.134766,0.095052,NaN,0.311523,0.619141,0.453125,0.622396,NaN,1,LDK
1178,1178,79218601,https://suumo.jp/ms/chuko/tokyo/sc_edogawa/nc_...,98.94,1994,8,1,東京都江戸川区北葛西４ [ ■ 周辺環境 ],西葛西駅,13,...,0.437500,0.162760,NaN,0.407552,0.710156,0.707031,0.519531,NaN,4,LDK
